# Applied Intelligent Systems - Lecture 7: Agentic AI in Data Science

In the previous lectures, we have explored the fundamentals of Language Models (LMs) and their applications for summarizing text, generating code, and answering questions. However, LMs so far are **passive tools** that require users to provide specific prompts to get the desired output. 

Agentic AI transforms LMs from passive, reactive knowledge systems into active systems capable of autonomously performing tasks, making decisions, and interacting with their environment. 
Instead of just answering questions, an AI Agent can plan and act to achieve specific goals, using tools and resources at its disposal.
An agent will break the task into smaller steps, decide which tools to use, and iteratively refine its approach based on feedback and results until the task is completed.

The transition from passive LMs to Agentic AI provides benefits, but also introduces new challenges and significant complexity compared to standard LM pipelines.

Our starting point is a standalone LM that can only generate text based on a given prompt. We will then enhance this LM with tool-use capabilities, allowing it to interact with external resources and perform more complex tasks autonomously.

After this lecture, you will be able to:
- Understand the concept of Agentic AI and how it differs from standard LMs.
- Implement a simple Agentic AI using the smolagents library and the Hugging Face Inference API.
- Add tool-use capabilities to your agent, enabling it to interact with external resources and perform complex tasks autonomously.
- Define and implement a custom tool for your agent to use in its decision-making process.

Let's get started!

---

## Variant 1: Standalone LM using the Hugging Face Inference API

As model, we will use the **Qwen 2.5-7B Instruct** model, which is a small but powerful language model that can generate high-quality text based on user prompts.

In [22]:
import dotenv

# Some globals first
HF_ACCESS_TOKEN = dotenv.get_key(".env", "HF_ACCESS_TOKEN")
MODEL = "Qwen/Qwen2.5-7B-Instruct"
USER_PROMPT = "Which wild bee species can I likely observe today in Kufstein?"

We will start with a simple implementation of a standalone LM using the Hugging Face Inference API:

In [23]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=HF_ACCESS_TOKEN)

In [24]:
response = client.chat.completions.create(
    model=MODEL, 
    messages=[{
        "role": "user", 
        "content": USER_PROMPT
    }],
    max_tokens=500
)

In [25]:
print(response.choices[0].message.content)

To determine which wild bee species you might observe in Kufstein, Austria, it's important to consider the local climate, habitat, and the time of year. Kufstein is located in the Tyrol region of Austria, which has a temperate climate with cold winters and mild summers.

Here are some common wild bee species you might observe in Kufstein:

1. **Bumblebees (Bombus spp.)**: These are large, fuzzy bees that are often seen in gardens and meadows. They are active from early spring to late autumn.

2. **Solitary Bees (e.g., Andrena, Halictus, Megachile spp.)**: These bees do not live in colonies but instead build their nests in the ground or in hollow stems. They are active throughout the spring and summer.

3. **Mining Bees (Andrena spp.)**: These are small to medium-sized bees that are often seen visiting flowers in early spring.

4. **Leafcutter Bees (Megachile spp.)**: These bees use leaves to construct their nests and are active during the summer.

5. **Mining Wasps (e.g., Sphecidae fam

As expected, the model attempts to answer the question based on its training data, but it does not have access to real-time information about the current date or location, so it cannot provide a more accurate answer to the user's question.

Next, we will enhance this LM with tool-use capabilities, allowing it to interact with external resources and perform more complex tasks autonomously.

## Variant 2: Agentic AI using the `smolagents` library

As the next step, we will implement a simple Agent capabale of performing a task autonomously. The agent will have access to a set of tools and a LM as its engine.

To incorporate tool-use capabilities, we will exchange the simple HF inference client with the `smolagents` library, which provides a framework for building agents that can interact with tools and resources.

The two main ways to build an agent in `smolagents` are Tool Calling Agents and Coding Agents `CodeAgent`.

A **Tool Calling Agent** is the "standard" approach, where the agent selects a tool and provides inputs in a structured format (e.g., JSON). This approach for building agents is reliable for simple tasks but less flexible for complex reasoning.

A **Coding Agent** is a more flexible approach, where the agent generates code to solve the task. This allows for more complex reasoning and decision-making, but it also requires the agent to be able to write correct code and handle errors effectively.

To build an agent, we need at least two elements:

- `tools`: a list of tools the agent has access to.
- `model`: an LLM that serves as the engine of the agent.

Tools can be downloaded from the Hugging Face Hub or other frameworks. We can also create **custom tools** by writing our own functions. This will be explained in a later section.

For the **model**, the framework provides distinct classes to connect to different LLM providers:

- `InferenceClientModel` is the default class, which connects to Hugging Face's serverless inference service, allowing developers to run models hosted on the Hugging Face Hub. I.e., the agent runs on Hugging Face infrastructure, and no local GPU is required. However, only a small number of free tokens are offered per month for experimentation and prototyping. Beyond that quota, usage becomes pay-as-you-go under provider rates.
- `HfApiModel` class also uses Hugging Face's free inference API to give access to open-source LLMs and other models without local hosting. Still, availability and cost depend on the compute requirements of the chosen model, and for medium to large LLMs or frequent use, you may quickly exhaust free credits and need to pay for additional usage. The `HfAPIModel` interface is older legacy method for running inference on Hugging Face from older `huggingface_hub` versions, while `InferenceClientModel` is the modern, fully supported API that provides consistent outputs, streaming, and model support.
- `LiteLLMModel` class allows users to choose from a list of 100+ proprietary LLM providers, like OpenAI, Anthropic, or Azure. Agents can be powered by models like GPT-4o or Claude 4.5 Sonnet, which often have superior performance for highly complex tasks. Using proprietary LLMs requires providing the API key, and there are costs associated with using these models.

In [26]:
from smolagents import ToolCallingAgent
from smolagents.models import InferenceClientModel

# Initialize the model
model = InferenceClientModel(
    model_id=MODEL, 
    api_key=HF_ACCESS_TOKEN
)

# Create an agent with the model, but no tools for now
agent = ToolCallingAgent(
    model=model, 
    tools=[]
)

# Run the agent with the user prompt
agent_response = agent.run(USER_PROMPT)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Which wild bee species can I likely observe today in Kufstein?                                                  │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-7B-Instruct ───────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Some common wild bee species you might observe in      │
│ Kufstein today include the European honey bee (Apis mellifera), bumblebees (Bombus spp.), mason bees (Osmia     │
│ spp.), and solitary bees such as leafcutter bees (Megachilidae family). However, the specific species you can   │
│ observe will depend on the time of year and weather conditions.'}                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Some common wild bee species you might observe in Kufstein today include the European honey bee (Apis
mellifera), bumblebees (Bombus spp.), mason bees (Osmia spp.), and solitary bees such as leafcutter bees 
(Megachilidae family). However, the specific species you can observe will depend on the time of year and weather 
conditions.

Final answer: Some common wild bee species you might observe in Kufstein today include the European honey bee (Apis
mellifera), bumblebees (Bombus spp.), mason bees (Osmia spp.), and solitary bees such as leafcutter bees 
(Megachilidae family). However, the specific species you can observe will depend on the time of year and weather 
conditions.

[Step 1: Duration 1.61 seconds| Input tokens: 1,017 | Output tokens: 95]

We can inspect the thinking process of the agent by calling `agent.replay()`, which shows the sequence of thoughts, actions, and tool calls made by the agent to arrive at its final answer. This is useful for debugging and understanding how the agent is reasoning through the problem.

In [27]:
agent.replay()

[19:28:19] Replaying the agent's steps:                                                               ]8;id=4361140;file:///Users/mase/Documents/Privat/Bewerbung/2026_03_FH_Kufstein/src/venv/lib/python3.11/site-packages/smolagents/memory.py\memory.py]8;;\:]8;id=4361141;file:///Users/mase/Documents/Privat/Bewerbung/2026_03_FH_Kufstein/src/venv/lib/python3.11/site-packages/smolagents/memory.py#256\256]8;;\

System prompt ─────────────────────────────────────────────────────────────────────────────────────────────────────
You are an expert assistant who can solve any task using tool calls. You will be given a task to solve as best you 
can.                                                                                                               
To do so, you have been given access to some tools.                                                                
                                                                                                                   
The tool call you write is an action: after the tool is executed, you will get the result of the tool call as an   
"observation".                                                                                                     
This Action/Observation can repeat N times, you should take several steps when needed.                             
                                                                                                                   
You can use the result of the previous action as input for the next action.                                        
The observation will always be a string: it can represent a file, like "image_1.jpg".                              
Then you can use it as input for the next action. You can do it for instance as follows:                           
                                                                                                                   
Observation: "image_1.jpg"                                                                                         
                                                                                                                   
Action:                                                                                                            
{                                                                                                                  
  "name": "image_transformer",                                                                                     
  "arguments": {"image": "image_1.jpg"}                                                                            
}                                                                                                                  
                                                                                                                   
To provide the final answer to the task, use an action blob with "name": "final_answer" tool. It is the only way to
complete the task, else you will be stuck on a loop. So your final output should look like this:                   
Action:                                                                                                            
{                                                                                                                  
  "name": [

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Which wild bee species can I likely observe today in Kufstein?                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Another way to inspect the agent's thinking process is to inspect the `agent.memory.steps`, which contains the raw sequence of thoughts, actions, and tool calls made by the agent during its reasoning process. This allows for a more detailed analysis of the agent's decision-making and can help identify any issues or areas for improvement in the agent's reasoning.

In [ ]:
agent.memory.steps

[TaskStep(task='Which wild bee species can I likely observe today in Kufstein?', task_images=None),
 ActionStep(step_number=1, timing=Timing(start_time=1778693283.410626, end_time=1778693285.019986, duration=1.6093599796295166), model_input_messages=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, content=[{'type': 'text', 'text': 'You are an expert assistant who can solve any task using tool calls. You will be given a task to solve as best you can.\nTo do so, you have been given access to some tools.\n\nThe tool call you write is an action: after the tool is executed, you will get the result of the tool call as an "observation".\nThis Action/Observation can repeat N times, you should take several steps when needed.\n\nYou can use the result of the previous action as input for the next action.\nThe observation will always be a string: it can represent a file, like "image_1.jpg".\nThen you can use it as input for the next action. You can do it for instance as follows:\n\nObservation: "

### Adding Custom Tools

So the LM realizes that it needs to gather current weather information and bee sightings to answer the user's question.
Let's see how we can tools to enable the agent to access real-time information and perform the necessary steps autonomously.

We can incorporate tools into our agent by providing a list of tools it can use. A simple example of a tool is `DuckDuckGo`, which allows the agent to perform web searches and retrieve real-time information from the internet.

Here, we will add custom tools:
- `get_current_date`: a tool that retrieves the current date.
- `get_weather`: a tool that retrieves the current weather information for a given location and date.
- `get_gbif_taxon_occurrences`: a tool that retrieves occurrence data for given taxa from a biodiversity database (GBIF).

The `smolagents` library allows creating custom tools using two main approaches:

1. Using the `@tool` decorator for simple function-based tools.
2. Creating a subclass of `Tool` for more complex functionality.

The `@tool` decorator is the recommended way to define simple tools. It simply requires writing a standard Python function with
- type hints, 
- a correctly formatted docstring, 
- and the `@tool` decorator. 

The framework parses this function to automatically generate the tool definition. This approach is much simpler than other frameworks that require complex class inheritance or JSON schema definitions to create custom tools.

In [29]:
from smolagents import tool

from datetime import date
import requests
import os
import pandas as pd


@tool
def get_current_date() -> str:
    """Returns the current date in ISO format (YYYY-MM-DD)."""
    return date.today().isoformat()


@tool
def get_weather(lat: float, lon: float, date: str = None) -> dict:
    """
    Fetches weather data for a location and date. If date is None, returns current weather.

    Args:
        lat: Latitude of the location (Kufstein: 47.58)
        lon:  Longitude of the location (Kufstein: 12.17)
        date: Optional date for historical weather (YYYY-MM-DD)
    
    Returns:
        A dictionary with weather information or an error message.
    """
    if date:
        # OpenMeteo Historical
        url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={date}&end_date={date}&daily=temperature_2m_max,precipitation_sum,windspeed_10m_max&timezone=auto"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return {
                "date": date,
                "temp_max": data['daily']['temperature_2m_max'][0],
                "rain": data['daily']['precipitation_sum'][0],
                "wind_max": data['daily']['windspeed_10m_max'][0],
                "source": "OpenMeteo Historical"
            }
    else:
        # OpenMeteo Current
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return {
                "date": get_current_date(),
                "temp": data['current_weather']['temperature'],
                "wind": data['current_weather']['windspeed'],
                "condition_code": data['current_weather']['weathercode'],
                "source": "OpenMeteo Current"
            }
    return "Error: Could not retrieve weather data."


@tool
def get_gbif_taxon_occurrences(lat: float, lon: float, taxon_keys: list = [], radius_km: int = 10, month: int = None) -> pd.DataFrame:
    """
    Retrieves occurrence records from GBIF for specified taxon keys within a radius of a location, optionally filtered by month.

    Args:
        lat (float): Latitude of the center point for the search (Kufstein: 47.58) 
        lon (float): Longitude of the center point for the search (Kufstein: 12.17)
        taxon_keys (list): List of GBIF taxon keys (int) for wild bee families
        radius_km (int): Search radius in kilometers
        month (int): Optional filter for month of observation (1-12)

    Returns:
        A pandas DataFrame with columns: species, date, lat, lon, or an error message if the API call fails.
    """

    # load from file if available
    gbif_occurrences_file = os.path.join("data", "gbif_sightings.csv")
    
    if os.path.exists(gbif_occurrences_file):
        df = pd.read_csv(gbif_occurrences_file)
        print("Loaded GBIF occurrences to DataFrame:")
        print(df.head())
        return df
            

    url = f"https://api.gbif.org/v1/occurrence/search?geoDistance={lat},{lon},{radius_km}km&limit=300"
    
    for key in taxon_keys:
        url += f"&taxonKey={key}"

    if month:
        url += f"&month={month}"
    
    response = requests.get(url)
    if not response.status_code == 200:
        return f"Error: {response.status_code} - {response.text}"

    results = response.json().get('results', [])
    
    sightings = []
    for record in results:
        sightings.append({
            "species": record.get('species', 'Unknown'),
            "date": record.get('eventDate', 'Unknown'),
            "lat": record.get('decimalLatitude', 'Unknown'),
            "lon": record.get('decimalLongitude', 'Unknown')
        })

    df = pd.DataFrame(sightings)
    print("Fetched GBIF occurrences and created DataFrame:")
    print(df.head())
    return df


@tool
def get_bee_taxon_keys() -> list:
    """
    Returns a list of GBIF taxon keys for seven wild bee families, i.e., 
    Apidae, Andrenidae, Colletidae, Halictidae, Megachilidae, Melittidae, Stenotritidae.
    """
    # These are the GBIF taxon keys for the respective families
    return [7901, 4334, 7905, 7908, 7911, 4345, 7916]

Now we can add these tools to our agent:

In [30]:
# Create an agent with the model, this time with our custom tools
agent = ToolCallingAgent(
    model=model, 
    tools=[ # add our tools to the agent
        get_current_date, 
        get_weather, 
        get_gbif_taxon_occurrences,
        get_bee_taxon_keys,
    ],
)

# Run the agent with the user prompt
agent_response = agent.run(USER_PROMPT)

print(agent_response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Which wild bee species can I likely observe today in Kufstein?                                                  │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-7B-Instruct ───────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_bee_taxon_keys' with arguments: {}                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |7901, 4334, 7905, 7908, 7911, 4345, 7916]

[Step 1: Duration 0.59 seconds| Input tokens: 2,107 | Output tokens: 18]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Based on the occurrence records, you are likely to     │
│ observe the following wild bee species in Kufstein today: Osmia bicolor, Bombus terrestris, Ceratina cyanea,    │
│ Osmia bicornis, Bombus pascuorum, Apis mellifera, Bombus lapidarius, Bombus hypnorum, Heriades truncorum,       │
│ Xylocopa violacea, Anthidium manicatum, Hylaeus communis, and Bombus pratorum.'}                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Based on the occurrence records, you are likely to observe the following wild bee species in Kufstein
today: Osmia bicolor, Bombus terrestris, Ceratina cyanea, Osmia bicornis, Bombus pascuorum, Apis mellifera, Bombus 
lapidarius, Bombus hypnorum, Heriades truncorum, Xylocopa violacea, Anthidium manicatum, Hylaeus communis, and 
Bombus pratorum.

Final answer: Based on the occurrence records, you are likely to observe the following wild bee species in Kufstein
today: Osmia bicolor, Bombus terrestris, Ceratina cyanea, Osmia bicornis, Bombus pascuorum, Apis mellifera, Bombus 
lapidarius, Bombus hypnorum, Heriades truncorum, Xylocopa violacea, Anthidium manicatum, Hylaeus communis, and 
Bombus pratorum.

[Step 3: Duration 1.70 seconds| Input tokens: 8,496 | Output tokens: 195]

Based on the occurrence records, you are likely to observe the following wild bee species in Kufstein today: Osmia bicolor, Bombus terrestris, Ceratina cyanea, Osmia bicornis, Bombus pascuorum, Apis mellifera, Bombus lapidarius, Bombus hypnorum, Heriades truncorum, Xylocopa violacea, Anthidium manicatum, Hylaeus communis, and Bombus pratorum.


In [31]:
agent.replay()

[19:29:19] Replaying the agent's steps:                                                               ]8;id=4361146;file:///Users/mase/Documents/Privat/Bewerbung/2026_03_FH_Kufstein/src/venv/lib/python3.11/site-packages/smolagents/memory.py\memory.py]8;;\:]8;id=4361147;file:///Users/mase/Documents/Privat/Bewerbung/2026_03_FH_Kufstein/src/venv/lib/python3.11/site-packages/smolagents/memory.py#256\256]8;;\

System prompt ─────────────────────────────────────────────────────────────────────────────────────────────────────
You are an expert assistant who can solve any task using tool calls. You will be given a task to solve as best you 
can.                                                                                                               
To do so, you have been given access to some tools.                                                                
                                                                                                                   
The tool call you write is an action: after the tool is executed, you will get the result of the tool call as an   
"observation".                                                                                                     
This Action/Observation can repeat N times, you should take several steps when needed.                             
                                                                                                                   
You can use the result of the previous action as input for the next action.                                        
The observation will always be a string: it can represent a file, like "image_1.jpg".                              
Then you can use it as input for the next action. You can do it for instance as follows:                           
                                                                                                                   
Observation: "image_1.jpg"                                                                                         
                                                                                                                   
Action:                                                                                                            
{                                                                                                                  
  "name": "image_transformer",                                                                                     
  "arguments": {"image": "image_1.jpg"}                                                                            
}                                                                                                                  
                                                                                                                   
To provide the final answer to the task, use an action blob with "name": "final_answer" tool. It is the only way to
complete the task, else you will be stuck on a loop. So your final output should look like this:                   
Action:                                                                                                            
{                                                                                                                  
  "name": [

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Which wild bee species can I likely observe today in Kufstein?                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
for step in agent.memory.steps[1:]:

    print(f"\n\n--- Step {getattr(step, 'step_number')} ---")

    for attr in ["code_action", "action_output", "tool_calls", "observations",]:
        print(f"\n{attr}")
        print(getattr(step, attr))




--- Step 1 ---

code_action
None

action_output
None

tool_calls
[ToolCall(name='get_bee_taxon_keys', arguments={}, id='call_s4l94mt0zfvakbuz9pb9xira')]

observations
[7901, 4334, 7905, 7908, 7911, 4345, 7916]


--- Step 2 ---

code_action
None

action_output
None

tool_calls
[ToolCall(name='get_gbif_taxon_occurrences', arguments={'lat': 47.58, 'lon': 12.17, 'taxon_keys': [], 'radius_km': 10}, id='call_4vfljnqr9jgxparbl2sb1l8v')]

observations
species                 date        lat        lon
0         Osmia bicolor  2026-05-03T12:52:51  47.647349  12.174019
1     Bombus terrestris           2025-05-01  47.648709  12.186470
2       Ceratina cyanea     2025-05-11T00:00  47.647408  12.174055
3        Osmia bicornis     2025-05-11T00:00  47.647308  12.174277
4      Bombus pascuorum           2025-05-13  47.551510  12.101389
5        Apis mellifera           2025-05-31  47.544200  12.291650
6        Apis mellifera     2025-05-21T13:13  47.534133  12.150612
7      Bombus pascuorum       

That is a more grounded reply, but as a Data Scientist, we want to have a more detailed and sophisticated answer. 

We can enhance our Agent's capabilities by writing and executing code in order to analyze the data it retrieves from the tools. 
This will allow the agent to not only gather information but also to process and interpret it to provide a more comprehensive answer to the user's question.

### Coding Agents

Differently from the `ToolCallingAgent` in `smolagents` that generates tool calls as JSON structures, a `CodeAgent` writes and executes Python code blocks. I.e., the `CodeAgent` writes a Python script that calls tools. This makes the agent more capable because it can perform math operations, process lists, and use logic (like if statements) within a single step.

In the following example, we create a new `InferenceClientModel` using a different LLM that is more suitable for Code Agents, `Qwen2.5-Coder-32B-Instruct`. Next, we define a new agent using `CodeAgent` and run the agent.

**Security of Code Agents**

Allowing AI agents to generate and execute Python code also introduces security risks, since a compromised model could take control of the host environment, run harmful commands, or access sensitive data.

- To address this, `smolagents` suggests, and in many production cases requires, *sandboxed execution* as the default and safest strategy. Production deployments typically use E2B, a cloud-based isolated environment that prevents any generated code from using the host machine.
- Developers who need local control can run agents inside *Docker containers*, where the AI agents work in an isolated environment to limit potential damage.
- In addition, `smolagents` allows developers to *restrict imports* using the `additional_authorized_imports` argument. By limiting the agent to safe libraries like `math`, `pandas`, `json`, etc., developers can significantly reduce the attack risks, though this does not provide the full isolation offered by a sandbox.

In [53]:
from smolagents import CodeAgent

# Create a client model using the Qwen2.5-Coder-32B-Instruct model
model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct", 
    token=HF_ACCESS_TOKEN
)

# This is our CodeAgent with our tools added
agent = CodeAgent(
    tools=[
        get_current_date, 
        get_weather, 
        get_gbif_taxon_occurrences,
        get_bee_taxon_keys
    ],
    add_base_tools=True,
    additional_authorized_imports=["numpy", "pandas"], # allow the agent to import numpy and pandas in its code
    model=model,
    max_steps=5 # limit the number of steps to prevent infinite loops during testing
)

The argument `add_base_tools=True` automatically includes default tools, such as Python Interpreter Tool, Final Answer Tool, etc., in addition to our custom tools.

In [55]:
# Run the agent
agent_response = agent.run(USER_PROMPT)

print(agent_response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Which wild bee species can I likely observe today in Kufstein?                                                  │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  today_date = get_current_date()                                                                                  
  print(f"Today's date is {today_date}.")                                                                          
  bee_taxon_keys = get_bee_taxon_keys()                                                                            
  print(f"Bee taxon keys are {bee_taxon_keys}.")                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Today's date is 2026-05-13.
Bee taxon keys are [7901, 4334, 7905, 7908, 7911, 4345, 7916].

Out: None

[Step 1: Duration 5.88 seconds| Input tokens: 2,496 | Output tokens: 114]

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import pandas as pd                                                                                              
                                                                                                                   
  # Convert the observation data into a DataFrame                                                                  
  occurrences_df = pd.DataFrame(occurrences)                                                                       
                                                                                                                   
  # Convert 'date' column to datetime, handling mixed formats                                                      
  occurrences_df['date'] = pd.to_datetime(occurrences_df['date'], errors='coerce')                                 
                                                                                                                   
  # Fill NaT values with the date part only                                                                        
  occurrences_df['date'] =                                                                                         
  occurrences_df['date'].fillna(pd.to_datetime(occurrences_df['date'].astype(str).str.split('T').str[0]))          
                                                                                                                   
  # Filter out domesticated honeybees and keep records from the last 5 years                                       
  filtered_occurrences = occurrences_df[                                                                           
      (occurrences_df['species'] != 'Apis mellifera') &                                                            
      (occurrences_df['date'].dt.year >= 2021)                                                                     
  ]                                                                                                                
                                                                                                                   
  # List unique wild bee species observed in May                                                                   
  unique_species = filtered_occurrences['species'][filtered_occurrences['date'].dt.month == 5].unique()            
  print(f"Unique wild bee species observed in Kufstein in the last 5 years in May: {unique_species}")              
                                                                                                                   
  final_answer(unique_species.tolist())                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Unique wild bee species observed in Kufstein in the last 5 years in May: <StringArray>
['Osmia bicolor', 'Bombus pascuorum']
Length: 2, dtype: str

Final answer: ['Osmia bicolor', 'Bombus pascuorum']

[Step 4: Duration 16.07 seconds| Input tokens: 15,930 | Output tokens: 928]

['Osmia bicolor', 'Bombus pascuorum']


I'm not satisfied with this answer.
Let's improve the prompt to encourage the agent to create a more detailed plan, use the tools effectively, and **model** the data it retrieves to provide a more comprehensive answer.

In [ ]:
enhanced_prompt = """
You are an expert Entomologist and Data Scientist specializing in Tyrolean biodiversity.
Your goal is to return the likelihood of observing a wild bee species in Kufstein based on current date, weather conditions, and historical sighting data along with corresponding weather data.

Get the current date and weather conditions for Kufstein using the provided tools.
Get a list of wild bee families, then get their historical sightings in the current month from GBIF for the area around Kufstein (radius of 10km).
Get the weather data for the days of the sightings.
Then analyze the relation between sightings and weather conditions using the provided Random Forest modeling tool.

Start by getting an overview on the tools available to you.
Then create a detailed plan on how to use them to solve the problem.
Where possible, use the provided tools to load pre-fetched data from files to save time instead of making API calls.
There is no need to edit, convert, or filter the data, just use the tools to retrieve the necessary information and then apply the modeling tool to analyze it.

Kufstein Location: lat=47.58, lon=12.17

Final Answer: [List of species with their probabilities]
"""

enhanced_prompt += "Original user question: " + USER_PROMPT

Also, let us help the agent by providing two other tools
- for loading prefetched weather data: `get_prefetched_weather_data()`
- one for running a statistical model: `run_statistical_model()`

In [59]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

@tool
def get_prefetched_weather_data() -> pd.DataFrame:
    """
    Fetches prefetched weather data for all the dates of the GBIF sightings from a local file.
    """
    weather_data_file = os.path.join("data", "weather_data.csv")
    if os.path.exists(weather_data_file):
        return pd.read_csv(weather_data_file)
    else:
        return "Error: Prefetched weather data file not found."


@tool
def analyze_bee_sightings_with_weather(
    bee_sightings_df: pd.DataFrame, 
    weather_df: pd.DataFrame, 
    current_temp: float,
    current_rain: float,
    current_wind: float,
    current_date: str,
    ) -> dict:
    """
    Fits a Random Forest Classifier to the prefetched bee sightings and weather data to model the likelihood of observing each bee species based on weather conditions and date.    
    
    Args:
        bee_sightings_df: DataFrame with columns 'species', 'date', 'lat', 'lon' for bee sightings returned by `get_gbif_taxon_occurrences`
        weather_df: DataFrame with columns 'date', 'temp_max', 'rain_sum', 'wind_max' for weather data returned by `get_prefetched_weather_data`
        current_temp: Current temperature
        current_rain: Current rain
        current_wind: Current wind
        current_date: Current date in ISO format (YYYY-MM-DD)
    """
    # equalize the date formats, ignore the time component if present
    bee_sightings_df['date'] = pd.to_datetime(bee_sightings_df['date'], errors='coerce', format="ISO8601").dt.date
    weather_df['date'] = pd.to_datetime(weather_df['date'], errors='coerce').dt.date
    merged_df = pd.merge(bee_sightings_df, weather_df, on="date")

    # convert date to day of year
    merged_df['doy'] = pd.to_datetime(merged_df['date']).dt.dayofyear

    # unique bee species and their counts
    species_list = merged_df['species'].value_counts()

    # define features and targets for modeling
    features = ['temp_max', 'rain_sum', 'wind_max', 'doy']
    x_data = merged_df[features]
    y = merged_df['species']

    # categorical encoding of the species names (=the target variable)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Fit a Random Forest Classifier
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(x_data, y_encoded)

    # predict probabilities using today's data
    doy = pd.to_datetime(current_date).dayofyear
    input_data = pd.DataFrame(
        [[current_temp, current_rain, current_wind, doy]], 
        columns=features
    )
    
    # Get probabilities for every species
    probs = rf.predict_proba(input_data)[0]
    
    # Combine with species names
    result = dict(zip(le.classes_, probs))
    
    # Sort by highest likelihood
    prediction = dict(sorted(result.items(), key=lambda item: item[1], reverse=True))

    print("Species Likelihoods:")
    for species, score in prediction.items():
        if score > 0: # Only show species with a chance
            print(f"{species}: {score:.1%}")
    return prediction

In [60]:
# Define a new CodeAgent with the enhanced tools
agent = CodeAgent(
    tools=[
        get_current_date, 
        get_weather, 
        get_gbif_taxon_occurrences,
        get_bee_taxon_keys,
        get_prefetched_weather_data, # we add this as a shortcut to load the weather data for the sightings without making API calls 
        analyze_bee_sightings_with_weather, # this is our modeling based on Random Forests
    ],
    add_base_tools=True,
    # additional_authorized_imports=["numpy", "pandas"],
    model=model
)

# Run the agent using the enhanced prompt
agent_response = agent.run(enhanced_prompt)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are an expert Entomologist and Data Scientist specializing in Tyrolean biodiversity.                        │
│ Your goal is to return the likelihood of observing a wild bee species in Kufstein based on current date,        │
│ weather conditions, and historical sighting data along with corresponding weather data.                         │
│                                                                                                                 │
│ Get the current date and weather conditions for Kufstein using the provided tools.                              │
│ Get a list of wild bee families, then get their historical sightings in the current month from GBIF for the     │
│ area around Kufstein (radius of 10km).                                                                          │
│ Get the weather data for the days of the sightings.                                                             │
│ Then analyze the relation between sightings and weather conditions using the provided Random Forest modeling    │
│ tool.                                                                                                           │
│                                                                                                                 │
│ Start by getting an overview on the tools available to you.                                                     │
│ Then create a detailed plan on how to use them to solve the problem.                                            │
│ Where possible, use the provided tools to load pre-fetched data from files to save time instead of making API   │
│ calls.                                                                                                          │
│ There is no need to edit or convert the data, just use the tools to retrieve the necessary information and then │
│ apply the modeling tool to analyze it.                                                                          │
│                                                                                                                 │
│ Kufstein Location: lat=47.58, lon=12.17                                                                         │
│                                                                                                                 │
│ Final Answer: [List of species with their probabilities\]                                                       │
│ Original user question: Which wild bee species can I likely observe today in Kufstein?                          │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Step 1: Get the current date                                                                                   
  current_date = get_current_date()                                                                                
                                                                                                                   
  # Step 2: Get current weather for Kufstein                                                                       
  kf_lat = 47.58                                                                                                   
  kf_lon = 12.17                                                                                                   
  weather = get_weather(lat=vf_lat, lon=kf_lon, date=None)                                                         
                                                                                                                   
  print(f"Current date: {current_date}")                                                                           
  print(weather)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Current date: 2026-05-13
{'date': '2026-05-13', 'temp': 13.5, 'wind': 8.4, 'condition_code': 3, 'source': 'OpenMeteo Current'}

Out: None

[Step 1: Duration 12.45 seconds| Input tokens: 2,962 | Output tokens: 299]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Analyze sightings and weather using the Random Forest model                                                    
  analysis_result = analyze_bee_sightings_with_weather(                                                            
      bee_sightings_df=sightings,                                                                                  
      weather_df=weather_subset,                                                                                   
      current_temp=weather['temp'],                                                                                
      current_rain=0.0,  # Assuming no rainfall as it's not provided                                               
      current_wind=weather['wind'],                                                                                
      current_date=current_date                                                                                    
  )                                                                                                                
                                                                                                                   
  # Convert values to standard float values                                                                        
  species_probabilities = {species: float(probability) for species, probability in analysis_result.items()}        
                                                                                                                   
  final_answer(species_probabilities)                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Species Likelihoods:
Bombus lapidarius: 32.4%
Bombus pascuorum: 18.9%
Bombus hypnorum: 17.4%
Nomada moeschleri: 6.0%
Ceratina cyanea: 5.9%
Osmia bicornis: 5.7%
Apis mellifera: 4.0%
Bombus pratorum: 4.0%
Bombus terrestris: 3.0%
Andrena apicata: 2.0%
Bombus bohemicus: 0.7%
Xylocopa violacea: 0.1%


Final answer: {'Bombus lapidarius': 0.32351587301587303, 'Bombus pascuorum': 0.1888095238095238, 'Bombus hypnorum':
0.1736111111111111, 'Nomada moeschleri': 0.06, 'Ceratina cyanea': 0.05885714285714285, 'Osmia bicornis': 
0.057428571428571426, 'Apis mellifera': 0.04, 'Bombus pratorum': 0.04, 'Bombus terrestris': 0.03, 'Andrena 
apicata': 0.02, 'Bombus bohemicus': 0.006666666666666666, 'Xylocopa violacea': 0.0011111111111111111, 'Anthidium 
manicatum': 0.0, 'Heriades truncorum': 0.0, 'Hylaeus communis': 0.0, 'Osmia bicolor': 0.0}

[Step 7: Duration 10.61 seconds| Input tokens: 43,036 | Output tokens: 1,574]

In [61]:
agent.replay()

[19:51:18] Replaying the agent's steps:                                                               ]8;id=4361152;file:///Users/mase/Documents/Privat/Bewerbung/2026_03_FH_Kufstein/src/venv/lib/python3.11/site-packages/smolagents/memory.py\memory.py]8;;\:]8;id=4361153;file:///Users/mase/Documents/Privat/Bewerbung/2026_03_FH_Kufstein/src/venv/lib/python3.11/site-packages/smolagents/memory.py#256\256]8;;\

System prompt ─────────────────────────────────────────────────────────────────────────────────────────────────────
You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you 
can.                                                                                                               
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can  
call with code.                                                                                                    
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and         
Observation sequences.                                                                                             
                                                                                                                   
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the 
tools that you want to use.                                                                                        
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with        
'<code>', and closed with '</code>'.                                                                               
During each intermediate step, you can use 'print()' to save whatever important information you will then need.    
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next    
step.                                                                                                              
In the end you have to return a final answer using the `final_answer` tool.                                        
                                                                                                                   
Here are a few examples using notional tools:                                                                      
---                                                                                                                
Task: "Generate an image of the oldest person in [

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ You are an expert Entomologist and Data Scientist specializing in Tyrolean biodiversity.                        │
│ Your goal is to return the likelihood of observing a wild bee species in Kufstein based on current date,        │
│ weather conditions, and historical sighting data along with corresponding weather data.                         │
│                                                                                                                 │
│ Get the current date and weather conditions for Kufstein using the provided tools.                              │
│ Get a list of wild bee families, then get their historical sightings in the current month from GBIF for the     │
│ area around Kufstein (radius of 10km).                                                                          │
│ Get the weather data for the days of the sightings.                                                             │
│ Then analyze the relation between sightings and weather conditions using the provided Random Forest modeling    │
│ tool.                                                                                                           │
│                                                                                                                 │
│ Start by getting an overview on the tools available to you.                                                     │
│ Then create a detailed plan on how to use them to solve the problem.                                            │
│ Where possible, use the provided tools to load pre-fetched data from files to save time instead of making API   │
│ calls.                                                                                                          │
│ There is no need to edit or convert the data, just use the tools to retrieve the necessary information and then │
│ apply the modeling tool to analyze it.                                                                          │
│                                                                                                                 │
│ Kufstein Location: lat=47.58, lon=12.17                                                                         │
│                                                                                                                 │
│ Final Answer: [List of species with their probabilities\]                                                       │
│ Original user question: Which wild bee species can I likely observe today in Kufstein?                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Agent output: ─────────────────────────────────────────────────────────────────────────────────────────────────────
### Thought:                                                                                                       
To solve this task, we need to use the given tools in a sequence that allows us to gather necessary information and
perform the analysis required. Here's a detailed plan:                                                             
                                                                                                                   
1. **Get the current date**: This will help us filter the bee sightings for the current month.                     
2. **Get the current weather conditions**: We'll need this information along with historical data to fit our model.
3. **Retrieve GBIF taxon keys for wild bees**: These keys are essential to filter bee sightings.                   
4. **Retrieve historical bee sightings**: For the current month, this will use the GBIF taxon keys to get sightings
within a 10km radius from Kufstein.                                                                                
5. **Retrieve historical weather data**: Corresponding weather data for the days of the sightings.                 
6. **Analyze the relation**: Use the `analyze_bee_sightings_with_weather` tool to analyze the collected data and   
get the likelihood of bee species observations on the current day's weather.                                       
                                                                                                                   
Let's start by getting the current date and weather conditions.                                                    
                                                                                                                   
### <code>                                                                                                         
# Step 1: Get the current date                                                                                     
current_date = get_current_date()                                                                                  
                                                                                                                   
# Step 2: Get current weather for Kufstein                                                                         
kf_lat = 47.58                                                                                                     
kf_lon = 12.17                                                                                                     
weather = get_weather(lat=vf_lat, lon=kf_lon, date=None)                                                           
                                                                                                                   
print(f"Current date: {current_date}")                                                                             
print(weather)                                                                                                     
</code>                                                                                                            

...

Agent output: ─────────────────────────────────────────────────────────────────────────────────────────────────────
### Thought:                                                                                                       
We have successfully retrieved the historical bee sightings dates and the corresponding historical weather data for
those dates. Now, we'll proceed to analyze the relation between sightings and weather conditions using the provided
`analyze_bee_sightings_with_weather` tool.                                                                         
                                                                                                                   
We'll prepare the necessary inputs:                                                                                
- `bee_sightings_df`: the `sightings` DataFrame                                                                    
- `weather_df`: the `weather_subset` DataFrame                                                                     
- `current_temp`: 13.5°C from the current weather data                                                             
- `current_rain`: 0.0mm (assuming no rainfall since it is not provided in the observation)                         
- `current_wind`: 8.4 km/h from the current weather data                                                           
- `current_date`: 2026-05-13                                                                                       
                                                                                                                   
### <code>                                                                                                         
# Analyze sightings and weather using the Random Forest model                                                      
analysis_result = analyze_bee_sightings_with_weather(                                                              
    bee_sightings_df=sightings,                                                                                    
    weather_df=weather_subset,                                                                                     
    current_temp=weather['temp'],                                                                                  
    current_rain=0.0,  # Assuming no rainfall as it's not provided                                                 
    current_wind=weather['wind'],                                                                                  
    current_date=current_date                                                                                      
)                                                                                                                  
                                                                                                                   
# Convert analysis result to a more readable format and extract species probabilities                              
species_probabilities = analysis_result.to_dict()                                                                  
                                                                                                                   
final_answer(species_probabilities)                                                                                
</code>                                                                                                            

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 6 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Agent output: ─────────────────────────────────────────────────────────────────────────────────────────────────────
### Thought:                                                                                                       
The error indicates that the result from the `analyze_bee_sightings_with_weather` function does not have a         
`to_dict` method. Instead, we should check the type of the result and handle it accordingly. If the result is a    
dictionary already, we can directly use it. If it's a DataFrame or some other type, we should convert it properly. 
                                                                                                                   
Since the result is a dictionary with NumPy float values, we can directly use it and convert the NumPy float64     
values to standard float values for readability.                                                                   
                                                                                                                   
### <code>                                                                                                         
import numpy as np                                                                                                 
                                                                                                                   
# Analyze sightings and weather using the Random Forest model                                                      
analysis_result = analyze_bee_sightings_with_weather(                                                              
    bee_sightings_df=sightings,                                                                                    
    weather_df=weather_subset,                                                                                     
    current_temp=weather['temp'],                                                                                  
    current_rain=0.0,  # Assuming no rainfall as it's not provided                                                 
    current_wind=weather['wind'],                                                                                  
    current_date=current_date                                                                                      
)                                                                                                                  
                                                                                                                   
# Convert NumPy float64 values to standard float values                                                            
species_probabilities = {species: float(probability) for species, probability in analysis_result.items()}          
                                                                                                                   
final_answer(species_probabilities)                                                                                
</code>                                                                                                            

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Agent output: ─────────────────────────────────────────────────────────────────────────────────────────────────────
### Thought:                                                                                                       
It seems that using `numpy` is not allowed. Since the values in the `analysis_result` dictionary are already in a  
numerical format, we can directly convert them to standard float values without importing `numpy`.                 
                                                                                                                   
### <code>                                                                                                         
# Analyze sightings and weather using the Random Forest model                                                      
analysis_result = analyze_bee_sightings_with_weather(                                                              
    bee_sightings_df=sightings,                                                                                    
    weather_df=weather_subset,                                                                                     
    current_temp=weather['temp'],                                                                                  
    current_rain=0.0,  # Assuming no rainfall as it's not provided                                                 
    current_wind=weather['wind'],                                                                                  
    current_date=current_date                                                                                      
)                                                                                                                  
                                                                                                                   
# Convert values to standard float values                                                                          
species_probabilities = {species: float(probability) for species, probability in analysis_result.items()}          
                                                                                                                   
final_answer(species_probabilities)                                                                                
</code>                                                                                                            